# Deliverable - Choices13k

This notebook predicts the choice rate (`bRate`) for each decision problem in Choices13k.
Each problem compares two lotteries (A and B), so the feature engineering focuses on both
absolute lottery statistics and B-minus-A comparisons.

The evaluation uses `GroupKFold` with `Problem` as the group id, so rows from the same
problem never appear in both train and validation folds.

At the end, models are compared with MAE (primary), RMSE, and R2, and the best
cross-validated MAE is reported in the final cell.


## 1) Setup and imports

This section imports all required libraries and sets the global experiment
configuration (seed, runtime profile, weighting mode, and XGBoost objective).
Keeping these settings in one place makes reruns reproducible and easier to audit.

Reproducibility note:
- Dependencies are listed in `requirements.txt`.
- Optional local setup (run once if needed): `%pip install -r requirements.txt`


In [ ]:

import json, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import GroupKFold, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, GradientBoostingRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.inspection import permutation_importance
from sklearn.neural_network import MLPRegressor

try:
    import xgboost as xgb
except ImportError:
    xgb = None

SEED = 42
np.random.seed(SEED)

EPS = 1e-9

# Profiles:
# - speed: quickest iterative cycle
# - balanced: good tradeoff
# - robust: most thorough, slowest
RUNTIME_PROFILE = "balanced"  # "speed" | "balanced" | "robust"

USE_GPU = False
WEIGHT_MODE = "sqrt"
XGB_OBJECTIVE = "reg:pseudohubererror"

# Speed toggles
RUN_FEATURE_SWEEP_DEFAULT = False
APPLY_SWEEP_BEST_K_DEFAULT = False

# Freeze a fast/high-performing setup from latest run
USE_FIXED_FEATURES = True
FIXED_20_FEATURES = [
    "EV_raw_Diff",
    "Hb",
    "pHb",
    "La",
    "Amb",
    "Lb",
    "EV_A_raw",
    "Ha",
    "LotShapeB",
    "EV_B_raw",
    "pHa",
    "LotNumB",
    "Amb_x_SD_Diff",
    "Amb_x_Range_Diff",
    "Feedback",
    "Mean_Loss_A",
    "P_Loss_A",
    "Mean_Gain_Diff",
    "Psych_EV2_B",
    "Entropy_Ratio",
]

USE_FIXED_XGB_PARAMS = False

# Optional fast rerun switch for quick checks
FAST_RERUN = False
CACHED_XGB_PARAMS = {
    "n_estimators": 1600,
    "max_depth": 7,
    "learning_rate": 0.012,
    "subsample": 0.75,
    "colsample_bytree": 0.8,
    "min_child_weight": 8.0,
    "reg_lambda": 0.3,
    "reg_alpha": 2.0,
    "gamma": 0.1,
}
FIXED_XGB_PARAMS = {
    "n_estimators": 1600,
    "max_depth": 7,
    "learning_rate": 0.012,
    "subsample": 0.75,
    "colsample_bytree": 0.8,
    "min_child_weight": 8.0,
    "reg_lambda": 0.3,
    "reg_alpha": 2.0,
    "gamma": 0.1,
}

if RUNTIME_PROFILE == "robust":
    FAST_MODE = False
    QUICK_MODE = False
elif RUNTIME_PROFILE == "balanced":
    FAST_MODE = True
    QUICK_MODE = False
else:
    FAST_MODE = True
    QUICK_MODE = True

ROBUST_MODE = (RUNTIME_PROFILE == "robust")
EARLY_STOPPING_ROUNDS = 30 if FAST_MODE else 50


In [ ]:
import os

assert os.path.exists("c13k_selections.csv"), "Missing c13k_selections.csv"
assert os.path.exists("c13k_problems.json"), "Missing c13k_problems.json"
print("Input files OK")


## 2) Feature engineering and dataset

The raw inputs are loaded and helper functions for feature extraction are defined.
Each problem produces a vector of statistics for A, B, and B-A, which are
then merged with experimental design variables from the selections table.

The output of this section is a clean modeling table X, the target y (bRate),
sample weights w, and the group labels used for GroupKFold.


### 2.1 Load data and safe division

This step loads the two input files:
- `c13k_selections.csv`
- `c13k_problems.json`

It also defines safe division helpers (`safe_div`, `soft_ratio`) to avoid
numerical instability when denominators are very small.


In [ ]:
def load_data():
    df = pd.read_csv("c13k_selections.csv")
    with open("c13k_problems.json", "r") as f:
        problems_dict = json.load(f)
    return df, problems_dict

def safe_div(a, b):
    return float(a / (b + EPS))


def soft_ratio(a, b, scale=1.0):
    return float(a / (abs(b) + scale))



### 2.2 Basic lottery statistics

Function: get_gamble_stats(outcomes)
Purpose: compute a standard set of descriptive statistics for a lottery.

What this block does:
1. Convert probabilities and payoffs into numpy arrays.
2. Compute EV = sum(p * x).
3. Compute variance and SD as the dispersion of outcomes.
4. Compute min, max, and range of payoffs.
5. Compute P_Gain, P_Loss, P_Zero based on payoff sign.
6. Compute EV_Gain and EV_Loss as weighted sums of positive/negative payoffs.
7. Compute Mean_Gain and Mean_Loss conditional on gain/loss.
8. Compute Mean_Abs for typical payoff magnitude.
9. Compute Max_Gain and Max_Loss as extreme outcomes.
10. Compute skewness and kurtosis using standardized z-scores.
11. Compute entropy as a measure of distribution uncertainty.
12. Return all statistics in a dictionary for later feature construction.


In [ ]:
def get_gamble_stats(outcomes):
    probs = np.array([o[0] for o in outcomes], dtype=float)
    probs = probs / (probs.sum() + EPS)
    pays  = np.array([o[1] for o in outcomes], dtype=float)

    ev = float(np.sum(probs * pays))
    var = float(np.sum(probs * (pays - ev) ** 2))
    sd = float(np.sqrt(var))

    mn = float(np.min(pays))
    mx = float(np.max(pays))
    rng = float(mx - mn)

    p_gain = float(np.sum(probs[pays > 0])) if np.any(pays > 0) else 0.0
    p_loss = float(np.sum(probs[pays < 0])) if np.any(pays < 0) else 0.0
    p_zero = float(np.sum(probs[pays == 0])) if np.any(pays == 0) else 0.0

    ev_gain = float(np.sum(probs[pays > 0] * pays[pays > 0])) if np.any(pays > 0) else 0.0
    ev_loss = float(np.sum(probs[pays < 0] * pays[pays < 0])) if np.any(pays < 0) else 0.0

    mean_gain = safe_div(ev_gain, p_gain) if p_gain > 0 else 0.0
    mean_loss = safe_div(ev_loss, p_loss) if p_loss > 0 else 0.0

    abs_pays = np.abs(pays)
    mean_abs = float(np.sum(probs * abs_pays))
    max_gain = float(np.max(pays[pays > 0])) if np.any(pays > 0) else 0.0
    max_loss = float(np.min(pays[pays < 0])) if np.any(pays < 0) else 0.0

    if sd > 0:
        z = (pays - ev) / sd
        skew = float(np.sum(probs * z**3))
        kurt = float(np.sum(probs * z**4))
    else:
        skew = 0.0
        kurt = 0.0

    pclip = np.clip(probs, EPS, 1.0)
    ent = float(-np.sum(pclip * np.log(pclip)))

    return {
        "EV": ev,
        "SD": sd,
        "Var": var,
        "Min": mn,
        "Max": mx,
        "Range": rng,
        "P_Gain": p_gain,
        "P_Loss": p_loss,
        "P_Zero": p_zero,
        "EV_Gain": ev_gain,
        "EV_Loss": ev_loss,
        "Mean_Gain": mean_gain,
        "Mean_Loss": mean_loss,
        "Mean_Abs": mean_abs,
        "Max_Gain": max_gain,
        "Max_Loss": max_loss,
        "Skew": skew,
        "Kurt": kurt,
        "Entropy": ent,
        "Num_Out": float(len(outcomes)),
    }



### 2.3 Subjective value and dominance probability

Two additional behavioral features are defined here:
- `psych_ev`: a prospect-style subjective value transform
- `prob_B_better_than_A`: probability that lottery B yields a larger payoff than A

These features are useful because human choices often deviate from purely
linear expected-value behavior.


In [ ]:
def psych_ev(outcomes, alpha=0.88, gamma=0.65):
    probs = np.array([o[0] for o in outcomes], dtype=float)
    probs = probs / (probs.sum() + EPS)
    pays  = np.array([o[1] for o in outcomes], dtype=float)

    subj_pays = np.sign(pays) * (np.abs(pays) ** alpha)
    probs_clipped = np.clip(probs, EPS, 1.0)
    weighted_probs = np.exp(-(-np.log(probs_clipped)) ** gamma)

    return float(np.sum(weighted_probs * subj_pays))

def prob_B_better_than_A(outA, outB):
    pA = np.array([o[0] for o in outA], dtype=float)
    pA = pA / (pA.sum() + EPS)
    xA = np.array([o[1] for o in outA], dtype=float)
    pB = np.array([o[0] for o in outB], dtype=float)
    pB = pB / (pB.sum() + EPS)
    xB = np.array([o[1] for o in outB], dtype=float)

    mat = (xB[:, None] > xA[None, :]).astype(float)
    return float(np.sum((pB[:, None] * pA[None, :]) * mat))



### 2.4 Feature rationale (what and why)

Each statistic is computed for A, for B, and for the difference (B-A).
The decision is comparative, so differences often carry more signal than
absolute levels. Ratios capture scale effects (e.g., relative risk), and
absolute differences capture magnitude regardless of sign.

| Feature family | What it captures and why it is included |
|---|---|
| EV | Expected value, central tendency of outcomes. |
| SD | Risk/variability around the mean. |
| Var | Alternative dispersion measure (sensitive to tails). |
| Min / Max | Worst and best outcome. |
| Range | Spread of possible outcomes. |
| P_Gain / P_Loss / P_Zero | Probabilities of gain, loss, and zero outcome. |
| EV_Gain / EV_Loss | Contribution of gains and losses to EV. |
| Mean_Gain / Mean_Loss | Typical size of gain/loss when it occurs. |
| Mean_Abs | Average absolute magnitude of outcomes. |
| Max_Gain / Max_Loss | Extreme gain or loss. |
| Skew / Kurt | Asymmetry and tail heaviness. |
| Entropy | Uncertainty/complexity of the lottery distribution. |
| Num_Out | Number of outcomes (structure/complexity). |
| Psych_EV / Psych_EV2 | Subjective value under prospect-style transforms. |
| CV (SD/|EV|) | Risk relative to expected value magnitude. |
| EV_Ratio / SD_Ratio / Range_Ratio | Relative scale of return and risk. |
| P_Gain_Ratio / P_Loss_Ratio | Relative probability of gain/loss. |
| Entropy_Ratio | Relative uncertainty. |
| P_B_better_A | Probability that B yields a higher payoff than A. |
| Amb, Corr, Feedback, LotShapeB, LotNumB, Block | Experimental design factors. |
| Ha/La/pHa and Hb/Lb/pHb (if present) | Raw lottery parameters. |
| EV_raw_Diff | EV difference using raw parameters. |
| Interaction terms | Design x risk/return effects (e.g., Amb*SD_Diff). |
| Abs differences | Magnitude of differences regardless of sign. |

Sample weights use n and bRate_std (if present) so more reliable
observations have higher influence during training.


### 2.4a Existing vs derived features

Existing features (directly from c13k_selections.csv):
- Amb, Corr, Feedback, LotShapeB, LotNumB, Block
- If present: Ha, La, pHa, Hb, Lb, pHb

Derived features (computed in this notebook):
- All lottery statistics for A, B, and B-A (EV, SD, Var, Min/Max, Range, etc.)
- Subjective value features (Psych_EV, Psych_EV2)
- Ratios (EV_Ratio, SD_Ratio, Range_Ratio, etc.)
- Interaction terms (e.g., Amb_x_SD_Diff, Feedback_x_EV_Diff)
- Absolute-difference features (e.g., EV_Diff_Abs)
- Dominance probability P_B_better_A


### 2.4b Feature naming convention

To make individual feature names readable, the notebook follows a simple naming scheme:
- `_A`: statistic of lottery A
- `_B`: statistic of lottery B
- `_Diff`: B minus A
- `_Abs`: absolute value of a difference
- `_Ratio`: relative scale (typically B vs A)
- `_x_`: interaction term with an experimental condition

Example: `Amb_x_SD_Diff` means ambiguity indicator multiplied by the SD difference (B-A).


### 2.5 Build the modeling dataset

Function: build_dataset(df, problems_dict)

What this block does:
1. Check for optional raw lottery columns and bRate_std.
2. Initialize lists for features, targets, weights, and group ids.
3. Define the stat keys to compute for A, B, and B-A.
4. Cache per-problem computations to avoid repeated work.
5. For each selection row:
   - Compute stats for A and B, plus subjective values.
   - Create A, B, and Diff features for each stat.
   - Add ratios, CV, and dominance probability.
   - Add experimental design variables.
   - Add raw parameters if present.
   - Add interaction and absolute-difference features.
   - Compute sample weight using n and bRate_std.
6. Assemble X, y, w, and groups into clean arrays and return.


In [ ]:
def build_dataset(df, problems_dict):
    raw_cols = ["Ha","La","pHa","Hb","Lb","pHb"]
    have_raw = all(c in df.columns for c in raw_cols)
    have_std = "bRate_std" in df.columns
    has_n = "n" in df.columns

    feats, y, w, groups = [], [], [], []

    stat_keys = [
        "EV","SD","Var","Min","Max","Range",
        "P_Gain","P_Loss","P_Zero",
        "EV_Gain","EV_Loss","Mean_Gain","Mean_Loss",
        "Mean_Abs","Max_Gain","Max_Loss",
        "Skew","Kurt","Entropy","Num_Out"
    ]

    problem_cache = {}

    for row in df.itertuples(index=False):
        pid = str(row.Problem)
        base = problem_cache.get(pid)
        if base is None:
            prob = problems_dict.get(pid)
            if prob is None:
                continue

            outA = prob["A"]
            outB = prob["B"]

            sA = get_gamble_stats(outA)
            sB = get_gamble_stats(outB)

            peA = psych_ev(outA)
            peB = psych_ev(outB)
            peA2 = psych_ev(outA, alpha=0.70, gamma=0.90)
            peB2 = psych_ev(outB, alpha=0.70, gamma=0.90)

            base = {}
            for k in stat_keys:
                base[f"{k}_A"] = sA[k]
                base[f"{k}_B"] = sB[k]
                base[f"{k}_Diff"] = sB[k] - sA[k]

            base["Psych_EV_A"] = peA
            base["Psych_EV_B"] = peB
            base["Psych_EV_Diff"] = peB - peA

            base["Psych_EV2_A"] = peA2
            base["Psych_EV2_B"] = peB2
            base["Psych_EV2_Diff"] = peB2 - peA2

            base["CV_A"] = safe_div(sA["SD"], abs(sA["EV"]) + 1e-6)
            base["CV_B"] = safe_div(sB["SD"], abs(sB["EV"]) + 1e-6)
            base["CV_Diff"] = base["CV_B"] - base["CV_A"]

            base["EV_Ratio"] = soft_ratio(sB["EV"], sA["EV"], scale=1.0)
            base["SD_Ratio"] = soft_ratio(sB["SD"], sA["SD"], scale=1.0)
            base["Range_Ratio"] = soft_ratio(sB["Range"], sA["Range"], scale=1.0)
            base["P_Gain_Ratio"] = soft_ratio(sB["P_Gain"], sA["P_Gain"], scale=1.0)
            base["P_Loss_Ratio"] = soft_ratio(sB["P_Loss"], sA["P_Loss"], scale=1.0)
            base["Entropy_Ratio"] = soft_ratio(sB["Entropy"], sA["Entropy"], scale=1.0)

            base["P_B_better_A"] = prob_B_better_than_A(outA, outB)

            problem_cache[pid] = base

        rec = base.copy()

        rec.update({
            "Amb": int(bool(row.Amb)),
            "Corr": int(row.Corr),
            "Feedback": int(bool(row.Feedback)),
            "LotShapeB": int(row.LotShapeB),
            "LotNumB": int(row.LotNumB),
            "Block": int(row.Block),
        })

        if have_raw:
            rec.update({
                "Ha": float(row.Ha),
                "La": float(row.La),
                "pHa": float(row.pHa),
                "Hb": float(row.Hb),
                "Lb": float(row.Lb),
                "pHb": float(row.pHb),
            })
            rec["EV_A_raw"] = float(row.pHa * row.Ha + (1 - row.pHa) * row.La)
            rec["EV_B_raw"] = float(row.pHb * row.Hb + (1 - row.pHb) * row.Lb)
            rec["EV_raw_Diff"] = rec["EV_B_raw"] - rec["EV_A_raw"]

        rec["Amb_x_SD_Diff"] = rec["Amb"] * rec["SD_Diff"]
        rec["Feedback_x_EV_Diff"] = rec["Feedback"] * rec["EV_Diff"]
        rec["Amb_x_Range_Diff"] = rec["Amb"] * rec["Range_Diff"]
        rec["Feedback_x_PGain_Diff"] = rec["Feedback"] * rec["P_Gain_Diff"]

        rec["EV_Diff_Abs"] = abs(rec["EV_Diff"])
        rec["SD_Diff_Abs"] = abs(rec["SD_Diff"])
        rec["Entropy_Diff_Abs"] = abs(rec["Entropy_Diff"])
        rec["Psych_EV_Diff_Abs"] = abs(rec["Psych_EV_Diff"])

        n = float(row.n) if has_n else 1.0
        if have_std:
            std = float(row.bRate_std)
            noise_factor = 1.0 / (std*std + 1e-4)
            noise_factor = min(noise_factor, 50.0)
            w.append(n * noise_factor)
        else:
            w.append(n)

        feats.append(rec)
        y.append(float(row.bRate))
        groups.append(pid)

    X = pd.DataFrame(feats)
    X = X.replace([np.inf, -np.inf], np.nan).fillna(0.0)

    y = np.array(y, dtype=float)
    w = np.array(w, dtype=float)
    groups = np.array(groups)
    return X, y, w, groups



### 2.6 Evaluation helpers

Functions: clip01(), metrics()

What this block does:
1. Clip predictions to [0, 1] because bRate is a probability.
2. Compute MAE, RMSE, and R2 for evaluation.
3. Return metrics for reporting and model comparison.


In [ ]:
def clip01(a):
    return np.clip(np.asarray(a, dtype=float), 0, 1)

def metrics(y_true, y_pred):
    y_pred = clip01(y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    r2 = r2_score(y_true, y_pred)
    return mae, rmse, r2



## 3) Models, tuning, and helpers

A small set of models that works well on tabular data and a
lightweight tuning strategy to keep runtime reasonable.

Why these models:
- XGBoost: strong baseline with good bias-variance tradeoff.
- RandomForest/ExtraTrees: robust ensembles capturing non-linearities.
- MLP: neural baseline for smooth non-linear patterns.


### 3.1 Models evaluated

At least three different ML methods and one neural network are evaluated:
- XGBoost (gradient-boosted trees)
- RandomForest (bagged trees)
- ExtraTrees (randomized trees)
- MLP (neural network)


### 3.2 Weighted training and GPU fallbacks

This block ensures models respect sample weights and remain stable even if
GPU acceleration fails or certain fit arguments are unsupported.

What this block does:
1. Detect GPU-related errors.
2. Provide compatibility for XGBoost fit arguments across versions.
3. Fit with sample weights and early stopping when available.
4. If GPU fails, fall back to CPU automatically.


In [ ]:

def _is_gpu_error(err):
    msg = str(err).lower()
    return "cuda" in msg or "gpu" in msg

def _xgb_fit_compat(model, X, y, fit_kwargs, sample_weight):
    try:
        model.fit(X, y, **fit_kwargs)
        return model
    except TypeError as e:
        msg = str(e)
        if "early_stopping_rounds" in msg:
            fit_kwargs.pop("early_stopping_rounds", None)
            try:
                model.fit(X, y, **fit_kwargs)
                return model
            except TypeError:
                pass
        if "sample_weight_eval_set" in msg:
            fit_kwargs.pop("sample_weight_eval_set", None)
            try:
                model.fit(X, y, **fit_kwargs)
                return model
            except TypeError:
                pass
        fit_kwargs.pop("eval_set", None)
        fit_kwargs.pop("sample_weight_eval_set", None)
        fit_kwargs.pop("early_stopping_rounds", None)
        try:
            model.fit(X, y, **fit_kwargs)
        except TypeError:
            model.fit(X, y, sample_weight=sample_weight)
        return model

def fit_with_weights(model, X, y, sample_weight, X_val=None, y_val=None, val_weight=None):
    module = model.__class__.__module__

    if module.startswith("xgboost"):
        fit_kwargs = {
            "sample_weight": sample_weight,
            "verbose": False
        }
        if X_val is not None and y_val is not None:
            fit_kwargs.update({
                "eval_set": [(X_val, y_val)],
                "early_stopping_rounds": EARLY_STOPPING_ROUNDS
            })
            if val_weight is not None:
                fit_kwargs["sample_weight_eval_set"] = [val_weight]
        try:
            return _xgb_fit_compat(model, X, y, fit_kwargs, sample_weight)
        except Exception as e:
            msg = str(e).lower()
            if _is_gpu_error(e) and USE_GPU:
                params = model.get_params()
                params.pop("device", None)
                params["tree_method"] = "hist"
                model = model.__class__(**params)
                return _xgb_fit_compat(model, X, y, fit_kwargs, sample_weight)
            if "objective" in msg and ("unknown" in msg or "not registered" in msg):
                params = model.get_params()
                params["objective"] = "reg:squarederror"
                model = model.__class__(**params)
                return _xgb_fit_compat(model, X, y, fit_kwargs, sample_weight)
            raise

    try:
        model.fit(X, y, sample_weight=sample_weight)
    except TypeError:
        model.fit(X, y)
    return model


### 3.3 Cross-validation helpers

These helpers generate GroupKFold splits and compute both CV MAE and OOF
predictions. This is the core evaluation logic used consistently for all
models.

What this block does:
1. Create group-aware folds if not provided.
2. Fit on train folds using weights.
3. Predict on validation folds and store in OOF vectors.
4. Return average MAE (and OOF predictions where needed).


In [ ]:
def cv_mae_sklearn(build_fn, params, X, y, w, groups, n_splits=3, scale=False, splits=None, X_values=None):
    if X_values is None:
        X_values = X.values

    if splits is None:
        gkf = GroupKFold(n_splits=n_splits)
        splits = list(gkf.split(np.zeros(len(y)), y, groups=groups))

    fold_maes = []

    for tr_i, val_i in splits:
        X_tr = X_values[tr_i]
        X_val = X_values[val_i]
        y_tr, y_val = y[tr_i], y[val_i]
        w_tr = w[tr_i]
        w_val = w[val_i]

        if scale:
            scaler = StandardScaler()
            Xtr_sc = scaler.fit_transform(X_tr)
            Xval_sc = scaler.transform(X_val)
        else:
            Xtr_sc = X_tr
            Xval_sc = X_val

        model = build_fn(params)
        model = fit_with_weights(model, Xtr_sc, y_tr, w_tr, X_val=Xval_sc, y_val=y_val, val_weight=w_val)

        pred = clip01(model.predict(Xval_sc))
        fold_maes.append(mean_absolute_error(y_val, pred))

    return float(np.mean(fold_maes)), float(np.std(fold_maes, ddof=1))

def cv_oof_sklearn(build_fn, params, X, y, w, groups, n_splits=5, scale=False, splits=None, X_values=None):
    if X_values is None:
        X_values = X.values

    if splits is None:
        gkf = GroupKFold(n_splits=n_splits)
        splits = list(gkf.split(np.zeros(len(y)), y, groups=groups))

    oof = np.zeros_like(y, dtype=float)

    for tr_i, val_i in splits:
        X_tr = X_values[tr_i]
        X_val = X_values[val_i]
        y_tr, y_val = y[tr_i], y[val_i]
        w_tr = w[tr_i]
        w_val = w[val_i]

        if scale:
            scaler = StandardScaler()
            Xtr_sc = scaler.fit_transform(X_tr)
            Xval_sc = scaler.transform(X_val)
        else:
            Xtr_sc = X_tr
            Xval_sc = X_val

        model = build_fn(params)
        model = fit_with_weights(model, Xtr_sc, y_tr, w_tr, X_val=Xval_sc, y_val=y_val, val_weight=w_val)
        oof[val_i] = model.predict(Xval_sc)

    return oof



### 3.4 Model parameter sampling and builders (MLP/RF/ET/GB/DT)

Each sample_* function draws hyperparameters from a compact, sensible range.
The build_* functions then construct the estimator with those parameters.

Key hyperparameters explained:
- n_estimators: number of trees (controls variance).
- max_depth/min_samples_leaf: tree complexity (controls overfitting).
- max_features: feature subsampling (reduces correlation among trees).
- learning_rate (GB): step size for boosting.
- hidden_layer_sizes/alpha (MLP): network capacity and regularization.

Note: depending on the sklearn version, MLPRegressor may ignore sample_weight.
This is why the report explicitly states that MLP may be unweighted.


In [ ]:
def sample_mlp_params(rng):
    if QUICK_MODE:
        hidden_options = [(32,), (64, 32)]
        max_iters = [120, 180]
    elif FAST_MODE:
        hidden_options = [(64, 32), (128, 64), (128, 64, 32), (256, 128)]
        max_iters = [250, 350]
    else:
        hidden_options = [(64, 32), (128, 64), (128, 64, 32), (256, 128), (256, 128, 64)]
        max_iters = [300, 450, 650]
    return {
        "hidden_layer_sizes": hidden_options[int(rng.randint(len(hidden_options)))],
        "alpha": float(rng.choice([1e-6, 1e-5, 1e-4, 1e-3])),
        "learning_rate_init": float(rng.choice([1e-4, 3e-4, 5e-4, 1e-3, 2e-3])),
        "batch_size": int(rng.choice([32, 64, 128])),
        "max_iter": int(rng.choice(max_iters)),
    }


def build_mlp(params):
    return MLPRegressor(
        random_state=SEED,
        activation="relu",
        solver="adam",
        early_stopping=True,
        n_iter_no_change=12,
        validation_fraction=0.1,
        **params
    )



def _sample_max_features(rng):
    options = ["sqrt", "log2", None, 0.6, 0.8]
    return options[int(rng.randint(len(options)))]

def sample_rf_params(rng):
    max_feat = _sample_max_features(rng)
    if FAST_MODE:
        n_estimators = rng.choice([300, 500, 800, 1200])
        max_depth = rng.choice([None, 10, 14, 18])
    else:
        n_estimators = rng.choice([500, 800, 1200, 1800])
        max_depth = rng.choice([None, 10, 14, 18, 24])
    return {
        "n_estimators": int(n_estimators),
        "max_depth": max_depth,
        "min_samples_leaf": int(rng.choice([1, 2, 3, 4])),
        "max_features": max_feat,
        "bootstrap": bool(rng.choice([True, False])),
    }


def build_rf(params):
    return RandomForestRegressor(
        random_state=SEED,
        n_jobs=1,
        **params
    )


def sample_et_params(rng):
    max_feat = _sample_max_features(rng)
    if FAST_MODE:
        n_estimators = rng.choice([300, 500, 800, 1200])
        max_depth = rng.choice([None, 10, 14, 18])
    else:
        n_estimators = rng.choice([500, 800, 1200, 1800])
        max_depth = rng.choice([None, 10, 14, 18, 24])
    return {
        "n_estimators": int(n_estimators),
        "max_depth": max_depth,
        "min_samples_leaf": int(rng.choice([1, 2, 3, 4])),
        "max_features": max_feat,
    }



def build_et(params):
    return ExtraTreesRegressor(
        random_state=SEED,
        n_jobs=1,
        **params
    )


def sample_gb_params(rng):
    return {
        "n_estimators": int(rng.choice([100, 200, 300])),
        "learning_rate": float(rng.choice([0.03, 0.05, 0.1])),
        "max_depth": int(rng.choice([2, 3, 4])),
        "min_samples_leaf": int(rng.choice([1, 2, 4])),
        "subsample": float(rng.choice([0.8, 1.0])),
    }


def build_gb(params):
    return GradientBoostingRegressor(
        loss="absolute_error",
        random_state=SEED,
        **params
    )


def sample_dt_params(rng):
    return {
        "max_depth": rng.choice([None, 6, 10, 14, 18]),
        "min_samples_leaf": int(rng.choice([1, 2, 4])),
        "min_samples_split": int(rng.choice([2, 5, 10])),
    }


def build_dt(params):
    return DecisionTreeRegressor(random_state=SEED, **params)





### 3.5 XGBoost sampling and builder

Depth, learning rate, subsampling, and regularization terms are sampled
to balance bias and variance. The builder uses a robust objective and
the hist tree method for speed. GPU is optional and disabled by default.


In [ ]:

def sample_xgb_params(rng):
    if QUICK_MODE:
        n_est_options = [700, 900, 1200]
        max_depth_options = [6, 7, 8]
        lr_options = [0.01, 0.015, 0.02]
    elif FAST_MODE:
        n_est_options = [900, 1200, 1600, 2000]
        max_depth_options = [6, 7, 8]
        lr_options = [0.01, 0.012, 0.015, 0.02]
    else:
        n_est_options = [900, 1200, 1600, 2000, 2600]
        max_depth_options = [4, 5, 6, 7, 8, 9]
        lr_options = [0.008, 0.01, 0.012, 0.015, 0.02]
    return {
        "n_estimators": int(rng.choice(n_est_options)),
        "max_depth": int(rng.choice(max_depth_options)),
        "learning_rate": float(rng.choice(lr_options)),
        "subsample": float(rng.choice([0.75, 0.8, 0.9, 1.0])),
        "colsample_bytree": float(rng.choice([0.75, 0.8, 0.9, 1.0])),
        "min_child_weight": float(rng.choice([2.0, 3.0, 5.0, 8.0])),
        "reg_lambda": float(rng.choice([0.3, 0.5, 1.0, 2.0, 5.0])),
        "reg_alpha": float(rng.choice([0.0, 0.05, 0.1, 0.5, 1.0])),
        "gamma": float(rng.choice([0.0, 0.05, 0.1, 0.3, 0.8])),
    }


def build_xgb(params):
    if xgb is None:
        raise ImportError("xgboost not installed. Run: pip install xgboost")
    base = {
        "random_state": SEED,
        "n_jobs": -1,
        "objective": XGB_OBJECTIVE,
        "eval_metric": "mae",
        "tree_method": "hist",
    }
    if USE_GPU:
        base["device"] = "cuda"
    return xgb.XGBRegressor(**base, **params)


### 3.6 Tuning and final training wrappers

The tuning wrapper performs a lightweight random search with GroupKFold.
The final training wrapper retrains the best model on all data and returns
the trained estimator (and scaler if used).


In [ ]:
def tune_sklearn_model(name, build_fn, sample_fn, X, y, w, groups, n_trials=20, n_splits=3, scale=False, splits=None):
    rng = np.random.RandomState(SEED)
    best = {"mae": 1e9, "std": None, "params": None}
    X_values = X.values

    for _ in range(n_trials):
        params = sample_fn(rng)
        mae, std = cv_mae_sklearn(
            build_fn, params, X, y, w, groups,
            n_splits=n_splits, scale=scale,
            splits=splits, X_values=X_values
        )
        if mae < best["mae"]:
            best = {"mae": mae, "std": std, "params": params}

    return best

def train_final_sklearn(build_fn, params, X, y, w, scale=False):
    scaler = None
    X_fit = X.values
    if scale:
        scaler = StandardScaler()
        X_fit = scaler.fit_transform(X.values)

    model = build_fn(params)
    # Κρατάμε το model που γυρνάει το fit_with_weights, γιατί μπορεί να έχει αλλάξει από GPU σε CPU
    model = fit_with_weights(model, X_fit, y, w)
    return model, scaler



### 3.7 Feature importance utilities

Feature importance is computed in three ways:
1. Built-in importance (tree models).
2. Absolute coefficients (linear models).
3. Permutation importance when no built-in scores exist.

The plot function also returns the top-K table used in the report.


In [ ]:

def plot_feature_importance(feature_names, scores, topk=15, title="Feature importance"):
    if scores is None:
        print("No importance scores available.")
        return None

    fi = (
        pd.DataFrame({"Feature": feature_names, "Score": scores})
        .sort_values("Score", ascending=False)
        .head(topk)
    )

    plt.figure(figsize=(9, 5))
    plt.barh(fi["Feature"][::-1], fi["Score"][::-1])
    plt.xlabel("Importance")
    plt.title(title)
    plt.tight_layout()
    plt.show()
    return fi

def permutation_importance_generic(model, X, y, feature_names, scaler=None, is_nn=False, n_repeats=5, max_samples=3000):
    if len(X) > max_samples:
        rng = np.random.RandomState(SEED)
        idx = rng.choice(len(X), size=max_samples, replace=False)
        X_use = X.iloc[idx]
        y_use = y[idx]
    else:
        X_use = X
        y_use = y

    class _Wrapper:
        def __init__(self, model, scaler, is_nn):
            self.model = model
            self.scaler = scaler
            self.is_nn = is_nn

        # sklearn>=1.6 validates that estimators implement fit.
        # underlying model already trained - inference-only wrapper
        def fit(self, X_in, y_in=None):
            return self

        def predict(self, X_in):
            Xp = self.scaler.transform(X_in) if self.scaler is not None else X_in
            if self.is_nn:
                return self.model.predict(Xp, verbose=0).ravel()
            return self.model.predict(Xp)

    def neg_mae(estimator, X_in, y_in):
        pred = estimator.predict(X_in)
        return -mean_absolute_error(y_in, clip01(pred))

    wrapper = _Wrapper(model, scaler, is_nn)
    result = permutation_importance(
        wrapper,
        X_use.values,
        y_use,
        scoring=neg_mae,
        n_repeats=n_repeats,
        random_state=SEED
    )
    return result.importances_mean


## 4) Training, cross-validation, and model selection

This section runs the full training loop: grouped splits, model-specific tuning,
OOF prediction generation, and metric-based comparison on the same CV protocol.


### 4.1 Train-test splitting and cross-validation

A plain random train/test split is not appropriate here, because multiple rows
belong to the same decision problem. To prevent leakage, `GroupKFold` is used
with `Problem` as the grouping key.

This produces out-of-fold (OOF) predictions for all rows and gives a stable,
fair estimate of generalization performance.


### 4.1a Hyperparameter tuning strategy

Hyperparameters are tuned separately for each model using random search over
predefined ranges (`sample_*_params`). Each candidate is evaluated with
`GroupKFold`, and selection is based on MAE.

Models tuned:
- XGBoost
- RandomForest
- ExtraTrees
- MLP


### 4.2 Load data and build the initial dataset

The next cell runs the full dataset construction pipeline and creates:
- `X`: feature matrix
- `y`: target (`bRate`)
- `w`: sample weights
- `groups`: problem ids for grouped CV


In [ ]:

df, problems_dict = load_data()
X, y, w, groups = build_dataset(df, problems_dict)

print(f"Samples: {len(X)} | Features: {X.shape[1]}")

if RUNTIME_PROFILE == "speed":
    MODELS_TO_TUNE = ["XGBoost"]
else:
    MODELS_TO_TUNE = ["XGBoost", "RandomForest", "ExtraTrees", "MLP"]


### 4.3 Configure trials, folds, and sample weights

This block sets the runtime budget (number of trials and folds) and applies the
chosen weighting strategy. In this run, `WEIGHT_MODE = "sqrt"` is used as a
balanced option between stability and influence of high-confidence rows.


In [ ]:

if RUNTIME_PROFILE == "speed":
    TRIALS_RF = 1
    TRIALS_ET = 1
    TRIALS_XGB = 3 if not USE_FIXED_XGB_PARAMS else 1
    TRIALS_MLP = 1
    TUNE_SPLITS = 2
    FINAL_SPLITS = 3
    PERM_REPEATS = 1
    PERM_SAMPLES = 500
    FEATURE_SELECTION_TOPK = None
elif RUNTIME_PROFILE == "balanced":
    TRIALS_RF = 3
    TRIALS_ET = 3
    TRIALS_XGB = 10
    TRIALS_MLP = 3
    TUNE_SPLITS = 3
    FINAL_SPLITS = 5
    PERM_REPEATS = 2
    PERM_SAMPLES = 1500
    FEATURE_SELECTION_TOPK = None
else:
    TRIALS_RF = 8
    TRIALS_ET = 8
    TRIALS_XGB = 24
    TRIALS_MLP = 10
    TUNE_SPLITS = 3
    FINAL_SPLITS = 5
    PERM_REPEATS = 4
    PERM_SAMPLES = 2500
    FEATURE_SELECTION_TOPK = None

if WEIGHT_MODE == "none":
    w_model = np.ones_like(w, dtype=float)
elif WEIGHT_MODE == "raw":
    w_model = w.astype(float)
elif WEIGHT_MODE == "sqrt":
    w_model = np.sqrt(np.clip(w, EPS, None))
elif WEIGHT_MODE == "clip10":
    w_model = np.clip(w, 1.0, 10.0)
else:
    raise ValueError("WEIGHT_MODE must be one of: none/raw/sqrt/clip10")

print(
    f"Weight mode: {WEIGHT_MODE} | "
    f"min={w_model.min():.3f}, median={np.median(w_model):.3f}, max={w_model.max():.3f}"
)
print(
    f"Profile={RUNTIME_PROFILE} | FAST={FAST_MODE}, QUICK={QUICK_MODE}, ROBUST={ROBUST_MODE} | "
    f"Objective={XGB_OBJECTIVE} | Models={MODELS_TO_TUNE} | "
    f"Trials (RF/ET/XGB/MLP) = {TRIALS_RF}/{TRIALS_ET}/{TRIALS_XGB}/{TRIALS_MLP} | "
    f"Splits (tune/final) = {TUNE_SPLITS}/{FINAL_SPLITS}"
)


### 4.4 Define model registry and check dependencies

Here the modeling registry is assembled (builder, parameter sampler, scaling
behavior, number of tuning trials per model), and required dependencies are
checked before training starts.


In [ ]:
MODEL_REGISTRY = {
    "RandomForest": {"build": build_rf, "sample": sample_rf_params, "scale": False, "trials": TRIALS_RF},
    "ExtraTrees": {"build": build_et, "sample": sample_et_params, "scale": False, "trials": TRIALS_ET},
    "XGBoost": {"build": build_xgb, "sample": sample_xgb_params, "scale": False, "trials": TRIALS_XGB},
    "MLP": {"build": build_mlp, "sample": sample_mlp_params, "scale": True, "trials": TRIALS_MLP},
}

missing = []
if "XGBoost" in MODELS_TO_TUNE and xgb is None:
    missing.append("xgboost")
if missing:
    raise ImportError("Missing packages: " + ", ".join(missing) + ". Run: pip install " + " ".join(missing))



### 4.5 Feature selection and reduced feature set

To keep runtime manageable, the notebook supports two modes:
- all features
- a compact preselected subset

The compact subset comes from prior importance scans and is used as a practical
speed/quality tradeoff. Automatic top-k filtering during CV is left disabled to
avoid optimistic estimates from feature filtering outside folds.


### 4.5a Compact feature set for speed

`FEATURE_SET = "compact"` uses a smaller list of high-signal features to cut runtime.
`FEATURE_SET = "all"` keeps the full engineered set.

For this submission, compact mode is used to keep execution time reasonable while
preserving strong CV performance.


In [ ]:

FEATURE_SET = "compact"  # "compact" or "all"
COMPACT_FEATURES = [
    "EV_raw_Diff",
    "Lb",
    "pHb",
    "EV_B_raw",
    "EV_A_raw",
    "Hb",
    "La",
    "Ha",
    "LotShapeB",
    "LotNumB",
    "Amb",
    "pHa",
    "Block",
    "EV_Diff_Abs",
    "EV_Diff",
    "P_B_better_A",
    "Entropy_Diff_Abs",
    "Num_Out_Diff",
    "Num_Out_B",
    "Amb_x_SD_Diff",
    "Kurt_A",
    "SD_Diff_Abs",
    "Psych_EV_Diff_Abs",
    "Psych_EV2_B",
    "Min_Diff",
    "Entropy_Diff",
    "Entropy_B",
    "EV_A",
    "Mean_Abs_A",
    "EV_B",
    "Feedback",
    "Amb_x_Range_Diff",
]

if USE_FIXED_FEATURES:
    selected_features = [f for f in FIXED_20_FEATURES if f in X.columns]
    missing = [f for f in FIXED_20_FEATURES if f not in X.columns]
    if missing:
        print(f"Warning: missing fixed features: {missing}")
elif FEATURE_SET == "compact":
    selected_features = [f for f in COMPACT_FEATURES if f in X.columns]
else:
    selected_features = X.columns.tolist()


In [ ]:
if xgb is not None and FEATURE_SELECTION_TOPK is not None and FEATURE_SELECTION_TOPK < X.shape[1]:
    selector_params = {
        "n_estimators": 900,
        "max_depth": 6,
        "learning_rate": 0.015,
        "subsample": 0.8,
        "colsample_bytree": 1.0,
        "min_child_weight": 2.0,
        "reg_lambda": 1.0,
        "reg_alpha": 2.0,
        "gamma": 0.3,
    }
    selector = build_xgb(selector_params)
    selector = fit_with_weights(selector, X.values, y, w_model)
    imp = np.asarray(getattr(selector, "feature_importances_", np.ones(X.shape[1])), dtype=float)
    order = np.argsort(imp)[::-1]
    topk = max(1, min(FEATURE_SELECTION_TOPK, X.shape[1]))
    selected_features = [X.columns[i] for i in order[:topk]]

X_model = X[selected_features].copy()
print(f"Using {X_model.shape[1]} selected features out of {X.shape[1]}")



### 4.5b Optional experiment (disabled by default)

This block is an optional long experiment for comparing:
- different feature counts (top-k)
- `GroupKFold` vs `KFold`

It is disabled by default because it is expensive and not required in the
standard run.


In [ ]:

from sklearn.model_selection import KFold, GroupKFold

RUN_SWEEP = bool(RUN_FEATURE_SWEEP_DEFAULT)
APPLY_SWEEP_BEST_K = bool(APPLY_SWEEP_BEST_K_DEFAULT and RUN_SWEEP and (not USE_FIXED_FEATURES))
SWEEP_APPLY_CV = "groupkfold"

SWEEP_K_LIST = [10, 20, 30, 40, 60]
SWEEP_SPLITS = 5
SWEEP_CV = ["groupkfold", "kfold"]
SWEEP_RANKING = "global"  # "global" (fast) or "foldwise" (slow but leakage-free)
SWEEP_MODEL = "xgb"  # "xgb" or "et"
SWEEP_N_ESTIMATORS = 400
SWEEP_MAX_DEPTH = 6
SWEEP_LEARNING_RATE = 0.05
SWEEP_ET_ESTIMATORS = 200

def _get_splits(strategy):
    if strategy == "groupkfold":
        return list(GroupKFold(n_splits=SWEEP_SPLITS).split(np.zeros(len(y)), y, groups=groups))
    if strategy == "kfold":
        return list(KFold(n_splits=SWEEP_SPLITS, shuffle=True, random_state=SEED).split(X))
    raise ValueError("Unknown strategy")

def _rank_features_global(X_in, y_in, w_in):
    if SWEEP_MODEL == "xgb" and xgb is not None:
        params = {
            "n_estimators": int(SWEEP_N_ESTIMATORS),
            "max_depth": int(SWEEP_MAX_DEPTH),
            "learning_rate": float(SWEEP_LEARNING_RATE),
            "subsample": 0.8,
            "colsample_bytree": 0.9,
            "min_child_weight": 3.0,
            "reg_lambda": 1.0,
            "reg_alpha": 0.0,
            "gamma": 0.0,
        }
        model = build_xgb(params)
        model = fit_with_weights(model, X_in.values, y_in, w_in)
        imp = np.asarray(getattr(model, "feature_importances_", np.ones(X_in.shape[1])), dtype=float)
    else:
        model = ExtraTreesRegressor(
            n_estimators=int(SWEEP_ET_ESTIMATORS),
            max_depth=14,
            min_samples_leaf=2,
            max_features="sqrt",
            random_state=SEED,
            n_jobs=1
        )
        model.fit(X_in.values, y_in, sample_weight=w_in)
        imp = model.feature_importances_
    order = np.argsort(imp)[::-1]
    return order

def _eval_k_global(order, k, splits):
    feats = X.columns[order[:k]]
    Xk = X[feats]
    if SWEEP_MODEL == "xgb" and xgb is not None:
        params = {
            "n_estimators": int(SWEEP_N_ESTIMATORS),
            "max_depth": int(SWEEP_MAX_DEPTH),
            "learning_rate": float(SWEEP_LEARNING_RATE),
            "subsample": 0.8,
            "colsample_bytree": 0.9,
            "min_child_weight": 3.0,
            "reg_lambda": 1.0,
            "reg_alpha": 0.0,
            "gamma": 0.0,
        }
        oof = cv_oof_sklearn(build_xgb, params, Xk, y, w_model, groups, n_splits=SWEEP_SPLITS, scale=False, splits=splits, X_values=Xk.values)
    else:
        params = {
            "n_estimators": int(SWEEP_ET_ESTIMATORS),
            "max_depth": 14,
            "min_samples_leaf": 2,
            "max_features": "sqrt",
        }
        oof = cv_oof_sklearn(build_et, params, Xk, y, w_model, groups, n_splits=SWEEP_SPLITS, scale=False, splits=splits, X_values=Xk.values)
    mae, rmse, r2 = metrics(y, oof)
    return mae, rmse, r2, list(feats)

def _eval_k_foldwise(k, splits):
    oof = np.zeros_like(y, dtype=float)
    for tr_i, val_i in splits:
        order = _rank_features_global(X.iloc[tr_i], y[tr_i], w_model[tr_i])
        feats = X.columns[order[:k]]
        X_tr = X.iloc[tr_i][feats]
        X_val = X.iloc[val_i][feats]
        if SWEEP_MODEL == "xgb" and xgb is not None:
            params = {
                "n_estimators": int(SWEEP_N_ESTIMATORS),
                "max_depth": int(SWEEP_MAX_DEPTH),
                "learning_rate": float(SWEEP_LEARNING_RATE),
                "subsample": 0.8,
                "colsample_bytree": 0.9,
                "min_child_weight": 3.0,
                "reg_lambda": 1.0,
                "reg_alpha": 0.0,
                "gamma": 0.0,
            }
            model = build_xgb(params)
            model = fit_with_weights(model, X_tr.values, y[tr_i], w_model[tr_i], X_val=X_val.values, y_val=y[val_i], val_weight=w_model[val_i])
            oof[val_i] = model.predict(X_val.values)
        else:
            params = {
                "n_estimators": int(SWEEP_ET_ESTIMATORS),
                "max_depth": 14,
                "min_samples_leaf": 2,
                "max_features": "sqrt",
            }
            model = build_et(params)
            model = fit_with_weights(model, X_tr.values, y[tr_i], w_model[tr_i], X_val=X_val.values, y_val=y[val_i], val_weight=w_model[val_i])
            oof[val_i] = model.predict(X_val.values)
    mae, rmse, r2 = metrics(y, oof)
    return mae, rmse, r2

if RUN_SWEEP:
    sweep_rows = []
    for strategy in SWEEP_CV:
        splits = _get_splits(strategy)
        if SWEEP_RANKING == "global":
            order = _rank_features_global(X, y, w_model)
            for k in SWEEP_K_LIST:
                mae, rmse, r2, _ = _eval_k_global(order, k, splits)
                sweep_rows.append({"cv": strategy, "ranking": "global", "k": k, "mae": mae, "rmse": rmse, "r2": r2})
        else:
            for k in SWEEP_K_LIST:
                mae, rmse, r2 = _eval_k_foldwise(k, splits)
                sweep_rows.append({"cv": strategy, "ranking": "foldwise", "k": k, "mae": mae, "rmse": rmse, "r2": r2})

    sweep_df = pd.DataFrame(sweep_rows).sort_values(["cv", "mae"]).reset_index(drop=True)
    display(sweep_df)

    for strategy in SWEEP_CV:
        sub = sweep_df[sweep_df["cv"] == strategy]
        if len(sub) > 0:
            best = sub.iloc[0]
            print(f"Best for {strategy}: K={int(best['k'])} | MAE={best['mae']:.6f}")

    sweep_best_k = None
    if APPLY_SWEEP_BEST_K and SWEEP_RANKING == "global":
        target_cv = SWEEP_APPLY_CV if SWEEP_APPLY_CV in SWEEP_CV else SWEEP_CV[0]
        sub = sweep_df[sweep_df["cv"] == target_cv].sort_values("mae")
        if len(sub) > 0:
            sweep_best_k = int(sub.iloc[0]["k"])
            order = _rank_features_global(X, y, w_model)
            selected_features = list(X.columns[order[:sweep_best_k]])
            X_model = X[selected_features].copy()
            print(f"Applied sweep best K={sweep_best_k} from {target_cv}. Using {X_model.shape[1]} features.")


### 4.5c Per-feature rationale table

The next cell builds a table with a short rationale for every feature used in
modeling. To keep this table complete and consistent, descriptions are assigned
with simple name-based rules (for example `EV_`, `SD_`, `_Diff`, `_Abs`).

This is documentation only; it does not affect model training.


In [ ]:
# Build a per-feature rationale table from feature names
feature_rationale_rules = [
    (r"^EV_", "Expected value related feature (central tendency)."),
    (r"^SD_", "Risk/variability feature (standard deviation)."),
    (r"^Var_", "Dispersion feature (variance)."),
    (r"^Min_", "Worst outcome feature."),
    (r"^Max_", "Best outcome feature."),
    (r"^Range_", "Outcome spread feature."),
    (r"^P_Gain", "Probability of gain feature."),
    (r"^P_Loss", "Probability of loss feature."),
    (r"^P_Zero", "Probability of zero outcome feature."),
    (r"^EV_Gain", "Contribution of gains to EV."),
    (r"^EV_Loss", "Contribution of losses to EV."),
    (r"^Mean_Gain", "Average gain conditional on gain."),
    (r"^Mean_Loss", "Average loss conditional on loss."),
    (r"^Mean_Abs", "Average absolute outcome magnitude."),
    (r"^Max_Gain", "Extreme gain size."),
    (r"^Max_Loss", "Extreme loss size."),
    (r"^Skew", "Asymmetry of outcomes."),
    (r"^Kurt", "Tail heaviness of outcomes."),
    (r"^Entropy", "Uncertainty/complexity of the lottery."),
    (r"^Num_Out", "Number of outcomes (structure)."),
    (r"^Psych_EV", "Subjective value (prospect-style transform)."),
    (r"^CV_", "Risk relative to EV magnitude (SD/|EV|)."),
    (r"^EV_Ratio", "Relative expected value (B vs A)."),
    (r"^SD_Ratio", "Relative risk (B vs A)."),
    (r"^Range_Ratio", "Relative spread (B vs A)."),
    (r"^P_Gain_Ratio", "Relative probability of gain (B vs A)."),
    (r"^P_Loss_Ratio", "Relative probability of loss (B vs A)."),
    (r"^Entropy_Ratio", "Relative uncertainty (B vs A)."),
    (r"^P_B_better_A", "Probability that B payoff exceeds A."),
    (r"^Amb$", "Ambiguity condition indicator."),
    (r"^Corr$", "Correlation condition indicator."),
    (r"^Feedback$", "Feedback condition indicator."),
    (r"^LotShapeB$", "Lottery shape of option B."),
    (r"^LotNumB$", "Number of outcomes in option B."),
    (r"^Block$", "Block index (experimental sequence)."),
    (r"^Ha$|^La$|^pHa$|^Hb$|^Lb$|^pHb$", "Raw lottery parameters (if present)."),
    (r"^EV_A_raw$|^EV_B_raw$|^EV_raw_Diff$", "Raw EV from parameters and its difference."),
    (r"_Diff$", "Difference between B and A for the same statistic."),
    (r"_A$|_B$", "Statistic for option A or B (absolute level)."),
    (r"_Abs$", "Absolute magnitude of the difference (sign removed)."),
    (r"_x_", "Interaction term between design and risk/return."),
]

import re
rows = []
if "selected_features" not in globals():
    if "X_model" in globals():
        selected_features = list(X_model.columns)
    elif "X" in globals():
        selected_features = list(X.columns)
    else:
        raise RuntimeError("selected_features not available yet")
for feat in selected_features:
    rationale = "(no match)"
    for pattern, desc in feature_rationale_rules:
        if re.search(pattern, feat):
            rationale = desc
            break
    rows.append({"feature": feat, "rationale": rationale})

feature_rationale_df = pd.DataFrame(rows)
display(feature_rationale_df)


### 4.6 Build CV splits and define OOF bagging for XGBoost

This step prepares grouped splits for tuning and final OOF evaluation.
For XGBoost, OOF predictions are averaged across multiple random seeds to
reduce variance and make the final estimate more stable.


In [ ]:

X_values = X_model.values
tune_splits = list(GroupKFold(n_splits=TUNE_SPLITS).split(np.zeros(len(y)), y, groups=groups))
final_splits = list(GroupKFold(n_splits=FINAL_SPLITS).split(np.zeros(len(y)), y, groups=groups))

if RUNTIME_PROFILE == "speed":
    XGB_OOF_SEEDS = [SEED]
elif RUNTIME_PROFILE == "balanced":
    XGB_OOF_SEEDS = [SEED, SEED + 1337, SEED + 2024]
else:
    XGB_OOF_SEEDS = [SEED, SEED + 1337, SEED + 2024, SEED + 4242, SEED + 9001]

print(f"XGBoost OOF seed ensemble: {XGB_OOF_SEEDS}")

def cv_oof_xgb_bagged(params, X_values, y, w, splits, seeds):
    oof = np.zeros_like(y, dtype=float)
    for tr_i, val_i in splits:
        X_tr = X_values[tr_i]
        X_val = X_values[val_i]
        y_tr, y_val = y[tr_i], y[val_i]
        w_tr = w[tr_i]
        w_val = w[val_i]

        pred_sum = np.zeros(len(val_i), dtype=float)
        for rs in seeds:
            model = build_xgb(params)
            try:
                model.set_params(random_state=int(rs))
            except Exception:
                pass
            model = fit_with_weights(model, X_tr, y_tr, w_tr, X_val=X_val, y_val=y_val, val_weight=w_val)
            pred_sum += model.predict(X_val)

        oof[val_i] = pred_sum / float(len(seeds))
    return oof


### 4.7 Tune models and build the results table

Each model is tuned with random search under the same `GroupKFold` setup.
After tuning, OOF predictions are generated and evaluated with MAE, RMSE, and R2.
The comparison table below is the main model-selection evidence.


In [ ]:

results = []
best_params = {}
oof_preds = {}

for name in MODELS_TO_TUNE:
    cfg = MODEL_REGISTRY[name]

    if name == "XGBoost" and FAST_RERUN:
        best = {"mae": np.nan, "std": None, "params": CACHED_XGB_PARAMS.copy()}
    elif name == "XGBoost" and USE_FIXED_XGB_PARAMS:
        best = {"mae": np.nan, "std": None, "params": FIXED_XGB_PARAMS.copy()}
    else:
        best = tune_sklearn_model(
            name=name,
            build_fn=cfg["build"],
            sample_fn=cfg["sample"],
            X=X_model, y=y, w=w_model, groups=groups,
            n_trials=cfg["trials"],
            n_splits=TUNE_SPLITS,
            scale=cfg["scale"],
            splits=tune_splits
        )

    if name == "XGBoost" and (not USE_FIXED_XGB_PARAMS) and (not FAST_RERUN) and RUNTIME_PROFILE != "speed":
        xgb_anchor_params = [
            {
                "n_estimators": 1600,
                "max_depth": 7,
                "learning_rate": 0.012,
                "subsample": 0.75,
                "colsample_bytree": 0.8,
                "min_child_weight": 8.0,
                "reg_lambda": 0.3,
                "reg_alpha": 2.0,
                "gamma": 0.1,
            },
            {
                "n_estimators": 1200,
                "max_depth": 7,
                "learning_rate": 0.012,
                "subsample": 0.8,
                "colsample_bytree": 0.8,
                "min_child_weight": 2.0,
                "reg_lambda": 2.0,
                "reg_alpha": 0.0,
                "gamma": 0.05,
            },
            {
                "n_estimators": 1200,
                "max_depth": 7,
                "learning_rate": 0.012,
                "subsample": 0.9,
                "colsample_bytree": 0.9,
                "min_child_weight": 3.0,
                "reg_lambda": 2.0,
                "reg_alpha": 1.0,
                "gamma": 0.05,
            },
        ]
        for p in xgb_anchor_params:
            mae_p, std_p = cv_mae_sklearn(
                cfg["build"], p, X_model, y, w_model, groups,
                n_splits=TUNE_SPLITS, scale=cfg["scale"],
                splits=tune_splits, X_values=X_values
            )
            if mae_p < best["mae"]:
                best = {"mae": mae_p, "std": std_p, "params": p}

    if name == "XGBoost":
        oof = cv_oof_xgb_bagged(best["params"], X_values, y, w_model, final_splits, XGB_OOF_SEEDS)
    else:
        oof = cv_oof_sklearn(
            build_fn=cfg["build"],
            params=best["params"],
            X=X_model, y=y, w=w_model, groups=groups,
            n_splits=FINAL_SPLITS,
            scale=cfg["scale"],
            splits=final_splits,
            X_values=X_values
        )

    mae, rmse, r2 = metrics(y, oof)
    results.append((name, mae, rmse, r2, best["params"]))
    oof_preds[name] = oof
    best_params[name] = best

res_df = (
    pd.DataFrame(results, columns=["Model", "CV_MAE", "CV_RMSE", "CV_R2", "Best_Params"])
    .sort_values("CV_MAE")
    .reset_index(drop=True)
)

display(res_df)


### 4.8 How to read the model comparison table

The table above reports cross-validated performance for each model.
- `CV_MAE`: primary metric (lower is better)
- `CV_RMSE`: penalizes larger errors more strongly (lower is better)
- `CV_R2`: explained variance (higher is better)

Model selection is based on the lowest `CV_MAE`.

## 5) Final model and results

After model selection, the best model is retrained on the full dataset for
interpretability, and its OOF metrics are reported.


In [ ]:

best_model = str(res_df.iloc[0]["Model"])
best_mae = float(res_df.iloc[0]["CV_MAE"])
print(f"Best model: {best_model} | MAE={best_mae:.6f}")

# Final training on full data
cfg = MODEL_REGISTRY[best_model]
final_model, final_scaler = train_final_sklearn(
    cfg["build"], best_params[best_model]["params"], X_model, y, w_model, scale=cfg["scale"]
)

# only one feature-importance output
fi = None
if hasattr(final_model, "feature_importances_"):
    scores = final_model.feature_importances_
    fi = plot_feature_importance(selected_features, scores, topk=15, title=f"{best_model} feature importance")
elif hasattr(final_model, "coef_"):
    scores = np.abs(np.asarray(final_model.coef_).ravel())
    fi = plot_feature_importance(selected_features, scores, topk=15, title=f"{best_model} |coef|")
else:
    scores = permutation_importance_generic(
        final_model,
        X_model,
        y,
        selected_features,
        scaler=final_scaler,
        is_nn=False,
        n_repeats=max(2, PERM_REPEATS),
        max_samples=max(1200, PERM_SAMPLES),
    )
    fi = plot_feature_importance(selected_features, scores, topk=15, title=f"Permutation importance ({best_model})")

if fi is not None:
    display(fi)

# OOF metrics for best model
best_oof = clip01(oof_preds[best_model])
mae, rmse, r2 = metrics(y, best_oof)
print(f"OOF metrics ({best_model}) -> MAE={mae:.6f} | RMSE={rmse:.6f} | R2={r2:.6f}")


### Which features are important and how

The top-15 feature-importance output summarizes which variables the final model
relied on most.

How to read it:
- For tree models, higher importance means the feature contributes more to splits.
- Importance indicates predictive reliance, not causal effect.
- The ranking is computed on the final model trained on all data (interpretability view).


## 6) Final result (MAE)

The final cell prints the CV metric table for all models and the best
cross-validated MAE from OOF predictions. Any lower MAE I found was due to data leakage. 


In [ ]:

print("Cross-validated metrics (all models):")
display(res_df[["Model", "CV_MAE", "CV_RMSE", "CV_R2"]])
print(f"Best MAE (CV): {best_mae:.6f}")


The time in my PC is about 14mins. I managed to make it under 4 mins but the MAE became bigger (0.080848)